# BLIP 训练教程

**一句话目标**：拿一个预训练好的 BLIP，在自己的数据集上微调，让它学会描述新领域的图片。

---

## 先问自己一个问题

如果你把一张 Pokemon 插画丢给 BLIP，它会说什么？

BLIP 预训练时见过大量**真实照片**，从来没见过这种二次元风格。  
它大概会说一些很笼统的话，比如 *"a drawing of a green animal"*，  
完全说不出 Pokemon 的特征。

**微调（Fine-tuning）就是：让模型见识新领域，学会说更准确的话。**

---

## 本 notebook 的结构

```
Step 0  准备环境
Step 1  看数据            ← 先搞清楚要学什么
Step 2  看 Baseline      ← 预训练模型现在有多拉跨
Step 3  理解训练原理      ← BLIP 到底在学什么？
Step 4  写训练代码        ← Dataset / DataLoader / 训练循环
Step 5  开始训练！
Step 6  看结果            ← 对比 before / after
```

## Step 0：准备环境

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset

import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"设备:    {device}")

## Step 1：看数据

我们用 [`lambdalabs/pokemon-blip-captions`](https://huggingface.co/datasets/lambdalabs/pokemon-blip-captions)：
- 833 张 Pokemon 风格插画，每张配一句英文描述
- 无需登录，约 50 MB，下载快
- 和自然照片风格差距很大，是验证「微调有没有用」的好实验场

> 下面的代码会先检查本地有没有缓存，有就直接读，没有再下载。  
> 如果下载中途断了，重新运行这个 cell 就好——HuggingFace datasets 支持断点续传。

In [ ]:
DATASET_NAME  = "lambdalabs/pokemon-blip-captions"
DATASET_CACHE = "./cache/datasets"
MODEL_NAME    = "Salesforce/blip-image-captioning-base"
MODEL_CACHE   = "./cache/models/blip-caption"

os.makedirs(DATASET_CACHE, exist_ok=True)
os.makedirs(MODEL_CACHE,   exist_ok=True)

# datasets 库把缓存存在 {cache_dir}/{name用___替换/}/ 下
slug = DATASET_NAME.replace("/", "___")
already_cached = any(Path(DATASET_CACHE).glob(f"{slug}*"))

if already_cached:
    print("[缓存命中] 直接从本地加载")
else:
    print("[开始下载] 约 50 MB，如果卡住检查代理后重新运行此 cell")

raw      = load_dataset(DATASET_NAME, cache_dir=DATASET_CACHE)
all_data = raw["train"]
print(f"\n加载完成，共 {len(all_data)} 条样本")
print(f"字段: {all_data.column_names}")
print(f"示例 caption: {all_data[0]['text']}")

In [ ]:
# 随机看 6 张，感受数据长什么样
idxs = np.random.choice(len(all_data), 6, replace=False)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, i in zip(axes.flatten(), idxs):
    sample = all_data[int(i)]
    ax.imshow(sample["image"])
    ax.axis("off")
    cap = sample["text"]
    ax.set_xlabel("\n".join(cap[j:j+42] for j in range(0, len(cap), 42)), fontsize=8)

plt.suptitle("数据集样本：Pokemon 插画 + 描述文字", fontsize=12)
plt.tight_layout()
plt.show()

## Step 2：看 Baseline——预训练模型有多拉跨

加载预训练 BLIP，直接对上面这类图生成描述，记录「还没学过时」的效果。  
训练结束后我们会用同一批图对比，直观感受提升。

> 同样先查缓存，本地有就不走网络。

In [ ]:
# 检查模型缓存
model_slug   = MODEL_NAME.replace("/", "--")
model_cached = Path(MODEL_CACHE).exists() and any(Path(MODEL_CACHE).glob(f"models--{model_slug}*"))

if model_cached:
    print("[缓存命中] 直接从本地加载模型")
else:
    print("[开始下载] 模型约 990 MB，如果卡住检查代理后重新运行")

processor = BlipProcessor.from_pretrained(MODEL_NAME, cache_dir=MODEL_CACHE)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_NAME, cache_dir=MODEL_CACHE).to(device)
model.eval()
print(f"模型加载完成，参数量: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
def predict(mdl, proc, images, device, max_length=50, num_beams=4):
    """对一批 PIL 图片生成描述"""
    mdl.eval()
    results = []
    for img in images:
        inp = proc(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = mdl.generate(**inp, max_length=max_length, num_beams=num_beams)
        results.append(proc.decode(ids[0], skip_special_tokens=True))
    return results


# 固定 6 张图，训练后用同一批做对比
COMPARE_IDXS   = np.random.choice(len(all_data), 6, replace=False)
compare_images = [all_data[int(i)]["image"].convert("RGB") for i in COMPARE_IDXS]
compare_refs   = [all_data[int(i)]["text"]                 for i in COMPARE_IDXS]

baseline_preds = predict(model, processor, compare_images, device)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, img, ref, pred in zip(axes.flatten(), compare_images, compare_refs, baseline_preds):
    ax.imshow(img)
    ax.axis("off")
    ax.set_xlabel(f"参考: {ref[:48]}\n预测: {pred[:48]}", fontsize=7.5)

plt.suptitle("Baseline：预训练模型（未微调）的效果", fontsize=12)
plt.tight_layout()
plt.show()

## Step 3：理解训练原理——BLIP 在学什么？

在写训练代码之前，先搞清楚「训练」到底是什么。

### BLIP Captioning 是「看图接龙」

```
图片  ──►  [Vision Encoder]  ──►  图像特征
                                        │
                                        ▼
""       ──►  [Text Decoder]  ──►  预测: "a"
"a"      ──►  [Text Decoder]  ──►  预测: "green"
"a green" ─►  [Text Decoder]  ──►  预测: "pokemon"
...
```

每一步，模型看着图片和已经有的词，预测「下一个词是什么」。

### 训练时做什么？

我们有正确答案（数据集里的 caption）。训练就是：

```
① 给模型：图片 + 正确 caption 作为上文
② 让它预测：每个位置的下一个词
③ 计算错误：预测词 vs 真实词  →  Cross-Entropy Loss
④ 反向传播：根据 loss 调整模型参数
⑤ 重复数千次：模型越来越准
```

> 注意：训练时我们把**正确答案**喂给模型做上文（Teacher Forcing），
> 不让模型自己一步步生成。这样训练更稳定、更快收敛。

### 体现在代码里

```python
outputs = model(
    pixel_values = 图片张量,
    input_ids    = caption 的 token 序列,   # 正确上文
    labels       = caption 的 token 序列,   # 正确答案（模型内部自动移位计算 loss）
)
loss = outputs.loss   # 模型帮我们算好了，直接用
```

## Step 4：写训练代码

### 4.1 全局配置

把所有超参数集中在一个 dataclass 里，要调参只改这里。

In [ ]:
@dataclass
class Config:
    # 数据
    val_ratio:  float = 0.15   # 15% 做验证集
    max_length: int   = 64     # caption 最长多少个 token

    # 训练
    batch_size:    int   = 4
    num_epochs:    int   = 5
    learning_rate: float = 5e-5
    weight_decay:  float = 0.01
    warmup_ratio:  float = 0.1    # 前 10% 步数做 LR warmup
    max_grad_norm: float = 1.0    # 梯度裁剪阈值
    use_amp:       bool  = True   # 混合精度，无 GPU 时自动关

    # 生成（评估时用）
    gen_max_length: int = 50
    num_beams:      int = 4

    # 保存
    output_dir: str = "./outputs/blip"


cfg = Config()

if not torch.cuda.is_available():
    cfg.use_amp = False
    print("无 GPU，已关闭混合精度")

print(cfg)

### 4.2 切分训练集和验证集

**一定要把训练集和验证集分开！**  

验证集只用来评估，绝对不能参与训练。  
如果你用训练集来评估，相当于考试时看答案——分数虚高，什么都说明不了。

In [ ]:
# seed=42 保证每次切分结果一致，方便复现
split      = all_data.train_test_split(test_size=cfg.val_ratio, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"训练集: {len(train_data)} 样本")
print(f"验证集: {len(val_data)} 样本")

### 4.3 Dataset 类

把原始数据（PIL Image + 字符串）转成模型需要的张量。

**重点：labels 里的 -100**

模型 caption 的长度不一样，我们会把短的 padding 到统一长度。  
但 padding 是凑数的，我们不希望 loss 去计算这些位置。  
解决办法：把 padding 位置的 label 设成 -100，PyTorch 的 loss 函数会自动跳过它们。

```
caption:   "a  green  pokemon  [PAD] [PAD]"
labels:    [23,  456,    789,   -100,  -100]   ← -100 不参与 loss
```

In [ ]:
class CaptionDataset(Dataset):
    def __init__(self, hf_data, processor, max_length):
        self.data       = hf_data
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample  = self.data[idx]
        image   = sample["image"].convert("RGB")
        caption = sample["text"]

        # processor 同时处理图像和文本，返回张量
        enc = self.processor(
            images=image,
            text=caption,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        pixel_values   = enc["pixel_values"].squeeze(0)

        # labels 和 input_ids 一样，但 padding 位置改成 -100
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
            "caption":        caption,   # 原始字符串，评估 BLEU 时用
        }


def collate_fn(batch):
    # 字符串不能 torch.stack，需要单独处理
    return {
        "pixel_values":   torch.stack([b["pixel_values"]   for b in batch]),
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels":         torch.stack([b["labels"]         for b in batch]),
        "caption":        [b["caption"] for b in batch],
    }


train_ds = CaptionDataset(train_data, processor, cfg.max_length)
val_ds   = CaptionDataset(val_data,   processor, cfg.max_length)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                          shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size,
                          shuffle=False, collate_fn=collate_fn, num_workers=2)

# 验证一下形状
b = next(iter(train_loader))
print("batch shapes:")
for k, v in b.items():
    print(f"  {k:15s}: {v.shape if hasattr(v, 'shape') else f'{len(v)} strings'}")

### 4.4 优化器 + 调度器

**AdamW**：Transformer 的标配优化器  
**Cosine + Warmup 调度**：LR 从 0 线性升到目标值，然后按余弦曲线慢慢降低

```
LR
  │      /\
  │     /  \___
  │    /        \____
  │___/              \___
  └──────────────────────→ steps
    warmup  cosine decay
```

为什么要 warmup？  
刚开始时模型参数随机，用大 LR 容易「跑飞」。先用小 LR 热身，等模型稳定了再加速。

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
)

total_steps  = len(train_loader) * cfg.num_epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# GradScaler：混合精度必须；use_amp=False 时 enabled=False 相当于空操作
scaler = GradScaler(enabled=cfg.use_amp)

print(f"总步数: {total_steps}，warmup: {warmup_steps} 步")
print(f"混合精度: {'开启' if cfg.use_amp else '关闭'}")

### 4.5 训练一个 Epoch

每个 step 的流程：

```
① forward  →  得到 loss（模型内部算 cross-entropy）
② backward →  算梯度
③ clip     →  梯度裁剪（防止梯度爆炸，稳定训练）
④ step     →  更新参数
⑤ schedule →  更新 LR（按步更新，不是按 epoch）
```

混合精度（`autocast` + `GradScaler`）让前向传播用 fp16 运行，更快更省显存。  
`use_amp=False` 时 `autocast` 是空操作，代码完全一样，不用单独写两套。

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, cfg, device, epoch):
    model.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{cfg.num_epochs}")
    for batch in pbar:
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()

        # ① forward
        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,          # 模型内部自动计算 cross-entropy loss
            )
            loss = outputs.loss

        # ② backward
        scaler.scale(loss).backward()

        # ③ clip：先 unscale 回 fp32，阈值才正确
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)

        # ④⑤ update
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.1e}")

    return total_loss / len(loader)

### 4.6 评估：BLEU-4

评估时切换到**生成模式**——不给模型正确答案，让它自己生成，然后和参考文本对比。  
BLEU 衡量「生成文本和参考文本有多少词组重叠」，是 captioning 任务的标准指标。

**关键：全程只用验证集，训练集数据绝对不出现在这里。**

In [ ]:
def evaluate(model, loader, processor, cfg, device):
    model.eval()
    refs, hyps = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="评估", leave=False):
            pixel_values = batch["pixel_values"].to(device)
            captions     = batch["caption"]        # 原始参考文本（来自验证集）

            # 生成模式：不传 labels，让模型自由生成
            gen_ids = model.generate(
                pixel_values=pixel_values,
                max_length=cfg.gen_max_length,
                num_beams=cfg.num_beams,
            )
            preds = processor.batch_decode(gen_ids, skip_special_tokens=True)

            for pred, ref in zip(preds, captions):
                hyps.append(pred.lower().split())
                refs.append([ref.lower().split()])  # 外层 list：BLEU 支持多条参考

    # method1 平滑：短句子修正，避免 BLEU 为 0 的极端情况
    bleu4 = corpus_bleu(refs, hyps, smoothing_function=SmoothingFunction().method1)
    return bleu4

### 4.7 保存最优模型

每次 BLEU 创新高，就保存一次「目前为止最好的模型」。  
用 `save_pretrained` 保存后，之后可以直接 `from_pretrained(路径)` 加载，非常方便。

In [ ]:
def save_best(model, processor, cfg):
    path = os.path.join(cfg.output_dir, "best_model")
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    processor.save_pretrained(path)
    print(f"  已保存最优模型 → {path}")

## Step 5：开始训练！

主循环把前面所有函数串起来：  
训练 → 评估 → 判断是否最优 → 保存。

**GPU 上参考时间**：5 epochs × ~2 分钟 ≈ 10 分钟  
**CPU 上**：很慢，建议先把 `cfg.num_epochs` 改成 1 跑通流程

In [ ]:
history   = {"train_loss": [], "val_bleu4": []}
best_bleu = 0.0

print(f"训练集 {len(train_ds)} | 验证集 {len(val_ds)} | {cfg.num_epochs} epochs")
print("=" * 55)

for epoch in range(1, cfg.num_epochs + 1):

    # ── 训练 ──────────────────────────────────────────────────
    loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, cfg, device, epoch
    )

    # ── 评估（只用验证集）──────────────────────────────────────
    bleu4 = evaluate(model, val_loader, processor, cfg, device)

    # ── 是否最优？ ─────────────────────────────────────────────
    is_best = bleu4 > best_bleu
    if is_best:
        best_bleu = bleu4   # ← 正确更新；不更新 = 永远以为自己是最优
        save_best(model, processor, cfg)

    flag = " ← 新高！" if is_best else ""
    print(f"Epoch {epoch:2d}  loss {loss:.4f}  BLEU-4 {bleu4:.4f}{flag}")

    history["train_loss"].append(loss)
    history["val_bleu4"].append(bleu4)

print(f"\n训练完成！最优 BLEU-4: {best_bleu:.4f}")

## Step 6：看结果

### 6.1 训练曲线

In [ ]:
epochs = list(range(1, cfg.num_epochs + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], marker="o", color="steelblue")
ax1.set(title="训练 Loss（应该下降）", xlabel="Epoch", ylabel="Loss")
ax1.grid(alpha=0.3)

ax2.plot(epochs, history["val_bleu4"], marker="o", color="darkorange")
ax2.set(title="验证集 BLEU-4（应该上升）", xlabel="Epoch", ylabel="BLEU-4")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_ep = history["val_bleu4"].index(max(history["val_bleu4"])) + 1
print(f"最优 BLEU-4: {max(history['val_bleu4']):.4f}  出现在 Epoch {best_ep}")

### 6.2 微调前 vs 微调后

加载刚保存的最优模型，对最开始那 6 张图重新生成描述，和 Baseline 并排对比。

In [ ]:
# 加载最优模型（直接读本地目录，不走网络）
best_path      = os.path.join(cfg.output_dir, "best_model")
best_proc      = BlipProcessor.from_pretrained(best_path)
best_model     = BlipForConditionalGeneration.from_pretrained(best_path).to(device)

finetuned_preds = predict(best_model, best_proc, compare_images, device)

# 每行展示一张图：左=微调前，右=微调后
n = len(compare_images)
fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
axes[0][0].set_title("微调前（Baseline）", fontsize=11, fontweight="bold")
axes[0][1].set_title("微调后（Best Model）", fontsize=11, fontweight="bold")

for row, (img, ref, base, fine) in enumerate(
    zip(compare_images, compare_refs, baseline_preds, finetuned_preds)
):
    for col, (ax, pred) in enumerate(zip(axes[row], [base, fine])):
        ax.imshow(img)
        ax.axis("off")
        ax.set_xlabel(
            f"参考: {ref[:50]}\n预测: {pred[:50]}",
            fontsize=7.5,
        )

plt.suptitle("微调前 vs 微调后效果对比", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## 复盘：这个 notebook 里的关键设计

| 做了什么 | 为什么这么做 |
|---|---|
| `train_test_split(seed=42)` | 固定随机种子保证复现；val 和 train 严格隔离 |
| `labels` 中 padding = -100 | loss 只算真实 token，padding 不贡献梯度 |
| Cosine + Warmup 调度 | 开始 LR 大容易跑飞，warmup 让训练更稳定 |
| `clip_grad_norm_` | 防止梯度爆炸，fine-tune 大模型时尤其重要 |
| `autocast` + `GradScaler` | fp16 前向更快省显存；GradScaler 防梯度下溢 |
| `best_bleu` 只在 is_best 时更新 | 真正追踪最优，而不是每轮都覆盖保存 |
| 评估用 `model.generate()` | 真实推理模式，和最终使用方式一致 |

---

**想进一步探索？**
- 把 `num_epochs` 加大，看 BLEU 还能涨多少
- 只训练 text decoder，冻结 vision encoder（在 optimizer 里只传 `model.text_decoder.parameters()`）
- 试试 LoRA / PEFT：用极少参数（~1%）达到接近全量微调的效果（`requirements.txt` 已有 `peft`）